In [1]:
# Importacion de librerias
import pandas as pd
import numpy as np
import openpyxl
import pyarrow
from pathlib import Path



In [7]:
# Setup
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Definir Rutas de archivos
RUTA_TRANSACCIONES = Path('/content/Transacciones.csv')
RUTA_PRECIOS = Path('/content/Precios.xlsx')
RUTA_SALIDA = Path('/content/Ventas_Integradas.parquet')

# Validación de archivos
print("Archivo Transacciones existe:", RUTA_TRANSACCIONES.exists())
print("Archivo Productos existe:", RUTA_PRECIOS.exists())



Archivo Transacciones existe: True
Archivo Productos existe: True


In [10]:
Transacciones = pd.read_csv(RUTA_TRANSACCIONES)
Precios = pd.read_excel(RUTA_PRECIOS)

print("Transacciones:", Transacciones.shape)
print("Precios:", Precios.shape)
Transacciones.head()


Transacciones: (10, 3)
Precios: (10, 3)


,ID_Producto,Cantidad,Fecha
0,1,2.0,1/3/2026
1,2,3.0,2/2/2026
2,3,1.0,23/5/2026
3,4,5.0,2/3/2026
4,5,NaN,3/3/2026


In [ ]:
Precios.head()

,ID_Producto,Nombre_producto,Precio_Unitario
0,1,Taza Ceramica Rosa,4500
1,2,Taza Chica Blanca,3000
2,3,Caterera Italiana,27000
3,4,Cucharitas,1000
4,5,Tetera,15700


In [11]:
#Exploracion Transacciones
Transacciones.info()

print("\nNulos por columna (Transacciones):")
print(Transacciones.isnull().sum())
print("\ndtype ID_Producto en Transacciones:", Transacciones["ID_Producto"].dtype)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ID_Producto  10 non-null     int64  
 1   Cantidad     9 non-null      float64
 2   Fecha        10 non-null     object 
dtypes: float64(1), int64(1), object(1)
memory usage: 372.0+ bytes

Nulos por columna (Transacciones):
ID_Producto    0
Cantidad       1
Fecha          0
dtype: int64

dtype ID_Producto en Transacciones: int64


In [12]:
#Exploracion Precios
Precios.info()
print("\nNulos por columna (Precios):")
print(Precios.isnull().sum())
print("dtype ID_Producto en Precios:      ", Precios["ID_Producto"].dtype)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ID_Producto      10 non-null     int64 
 1   Nombre_producto  10 non-null     object
 2   Precio_Unitario  10 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 372.0+ bytes

Nulos por columna (Precios):
ID_Producto        0
Nombre_producto    0
Precio_Unitario    0
dtype: int64
dtype ID_Producto en Precios:       int64


In [13]:
#Correccion a Fecha

print(Transacciones["Fecha"].sample(10, random_state=3).tolist())

Transacciones["Fecha"] = pd.to_datetime(Transacciones["Fecha"], format="mixed", dayfirst=True)

print(Transacciones["Fecha"].dtype)
Transacciones["Fecha"].head()


['6/4/2026', '3/3/2026', '2/2/2026', '23/5/2026', '5/9/2026', '20/6/2026', '27/7/2026', '1/3/2026', '2/3/2026', '30/8/2026']
datetime64[ns]


,Fecha
0,2026-03-01
1,2026-02-02
2,2026-05-23
3,2026-03-02
4,2026-03-03


In [14]:
#Tratamiento de nulos
nulos_antes = Transacciones["Cantidad"].isnull().sum()
Transacciones = Transacciones.dropna(subset=["Cantidad"])


Transacciones["Cantidad"] = Transacciones["Cantidad"].astype(int)

print(f"\nFilas eliminadas por 'Cantidad' nula: {nulos_antes}")
print(f"Filas restantes en Transacciones: {len(Transacciones)}")


Filas eliminadas por 'Cantidad' nula: 1
Filas restantes en Transacciones: 9


In [15]:
#Merge

Ventas = pd.merge(Transacciones, Precios, on="ID_Producto", how="left", indicator=True)


print(Ventas["_merge"].value_counts())
display(Ventas.head())

_merge
both          9
left_only     0
right_only    0
Name: count, dtype: int64


,ID_Producto,Cantidad,Fecha,Nombre_producto,Precio_Unitario,_merge
0,1,2,2026-03-01,Taza Ceramica Rosa,4500,both
1,2,3,2026-02-02,Taza Chica Blanca,3000,both
2,3,1,2026-05-23,Caterera Italiana,27000,both
3,4,5,2026-03-02,Cucharitas,1000,both
4,6,3,2026-04-06,Filtro Cafe,4300,both


In [38]:
#Crear Columna total ventas

Ventas["total_venta"] = Ventas["Cantidad"] * Ventas["Precio_Unitario"]

In [42]:
#Control Columna
assert "total_venta" in Ventas.columns
assert (Ventas["total_venta"] == Ventas["Cantidad"] * Ventas["Precio_Unitario"]).all()
print("OK")

display(Ventas[["ID_Producto", "Cantidad", "Precio_Unitario", "total_venta"]].head())

OK


,ID_Producto,Cantidad,Precio_Unitario,total_venta
0,1,2,4500,9000
1,2,3,3000,9000
2,3,1,27000,27000
3,4,5,1000,5000
4,6,3,4300,12900


In [44]:
#Verificaciones de nulos, duplicados, montos negativos y fecha
assert Ventas.isnull().sum().sum() == 0
assert Ventas.duplicated().sum() == 0
assert (Ventas["total_venta"] >= 0).all()
assert pd.api.types.is_datetime64_any_dtype(Ventas["Fecha"])

print("Validaciones OK")
print(f"Resumen final: {len(Ventas)} transacciones por un total de ${Ventas['total_venta'].sum():,.0f}")

Validaciones OK
Resumen final: 9 transacciones por un total de $155,900


In [46]:

# Exportación a archivo Parquet
print(f"Archivo exportado: {RUTA_SALIDA} ({RUTA_SALIDA.stat().st_size / 1024:.1f} KB)" if RUTA_SALIDA.exists() else "Todavía no exportaste el archivo")


Archivo exportado: /content/Ventas_Integradas.parquet (5.1 KB)


In [47]:

# Chequeo final
assert RUTA_SALIDA.exists(), "No se generó el archivo parquet"
verificacion = pd.read_parquet(RUTA_SALIDA)
assert len(verificacion) == len(Ventas)
print("🎉 ¡Pre-entrega resuelta! El dataset consolidado está listo.")

🎉 ¡Pre-entrega resuelta! El dataset consolidado está listo.
